# Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import datetime
from pyquaternion import Quaternion
import plotly.express as px
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
import argparse

# Utilities

In [ ]:
############### General ###############
def remove_outlier (arr):
  mean = np.mean(arr, axis=0)
  std = np.std(arr, axis=0)
  final_list = [x for x in arr if (x > mean - 2 * std)]
  final_list = [x for x in final_list if (x < mean + 2 * std)]
  return final_list

############### Euler ###############
def euler_diff (arr, time):
  diff = []
  for i in range(len(arr)-1):
    diff.append((np.array(arr[i+1])-np.array(arr[i]))/time)
  return diff

def euler_diff_cir (arr, time):
  diff = []
  for i in range(len(arr)-1):
    diff.append(euler_diff_cir_array(arr[i], arr[i+1])/time)
  return diff
  
def euler_diff_cir_array (arr1, arr2):
  res = np.array([0,0,0])
  if (len(arr1) != len(arr2)):
    print("Error: length of two arrays are not equal")
    return
  for i in range(len(arr1)):
    res[i] = euler_diff_cir_element(arr1[i], arr2[i])
  return res

def euler_diff_cir_array_1d (arr1, arr2):
  res = []
  if (len(arr1) != len(arr2)):
    print("Error: length of two arrays are not equal")
    return
  for i in range(len(arr1)):
    res.append(euler_diff_cir_element(arr1[i], arr2[i]))
  return res

def euler_diff_cir_element (a, b):
  if (b - a > 180):
    return (b-360)-a
  elif (b - a < -180):
    return b-(a-360)
  else:
    return b-a

def euler_distance (arr, threshold = 0):
  sum = 0
  for i in range(len(arr)-1):
    if abs(arr[i+1]-arr[i]) > threshold:
      sum += abs(arr[i+1]-arr[i])
  return sum

def euler_circle (arr):
  circle = []
  for i in range(len(arr)):
    if (arr[i] > 180):
      circle.append(arr[i]-360)
    else:
      circle.append(arr[i])
  return circle

def euler_circle_neg2pos (arr):
  circle = []
  for i in range(len(arr)):
    if (arr[i] < 0):
      circle.append(arr[i]+360)
    else:
      circle.append(arr[i])
  return circle

def euler_negative (arr):
  neg = []
  for i in range(len(arr)):
    neg.append(-arr[i])
  return neg

############### Quaternion ###############
def quaternion_distance (arr, threshold = 0):
  sum = 0
  for i in range(len(arr)-1):
    if abs(Quaternion.absolute_distance(Quaternion(arr[i]), Quaternion(arr[i+1]))) > threshold:
      sum += Quaternion.absolute_distance(Quaternion(arr[i]), Quaternion(arr[i+1]))
  return sum

def quaternion_diff_abs (arr, time):
  diff = []
  for i in range(len(arr)-1):
    diff.append(Quaternion.absolute_distance(Quaternion(arr[i]), Quaternion(arr[i+1]))/time)
  return diff

def quaternion_diff (arr, time):
  diff = []
  for i in range(len(arr)-1):
    diff.append(Quaternion.distance(Quaternion(arr[i]), Quaternion(arr[i+1]))/time)
  return diff

In [ ]:
def flatten_list (arr):
  flattened_list = []
  for sublist in arr:
    if hasattr(sublist, '__iter__') and not isinstance(sublist, str):  # Check if it's iterable and not a string
      flattened_list.extend(sublist)
    else:
      flattened_list.append(sublist)  # Handle non-iterables by appending directly
  return flattened_list

In [ ]:
fov_list = [
  "90", "110", "130", "150", "170"
]

# Files

In [ ]:
userList = [
  'P001',
  'P003',
  'P004',
  'P007',
  'P011',
  'P012',
  'P014',
  'P016',
  'P017',
  'P020',
  'P021',
  'P022',
  'P023',
  'P024',
  'P031'
]

# Data Input

In [ ]:
rightHandAngle_eu_set_90 = []
leftHandAngle_eu_set_90 = []
rightHandAngle_eu_set_110 = []
leftHandAngle_eu_set_110 = []
rightHandAngle_eu_set_130 = []
leftHandAngle_eu_set_130 = []

for userIdx in range(len(userList)):
  path = '../data/uist24/pilot/'
  filename = 'LOG_Pilot(RE)_' + userList[userIdx] + '.txt'
  username = userList[userIdx]
  print("🤟 processing user", username)

  fov_index = 0
  headAngle_qu = [np.array([[0,0,0,0]])] * 5
  headAngle_eu = [np.array([[0,0,0]])] * 5
  rightHandPosition = [np.array([[0,0,0]])] * 5
  leftHandPosition = [np.array([[0,0,0]])] * 5
  rightHandAngle_qu = [np.array([[0,0,0,0]])] * 5
  leftHandAngle_qu = [np.array([[0,0,0,0]])] * 5
  rightHandAngle_eu = [np.array([[0,0,0]])] * 5
  leftHandAngle_eu = [np.array([[0,0,0]])] * 5
  position = [np.array([[0,0,0]])] * 5

  completeFileName = path + filename

  f = open(completeFileName, 'r')

  count = 0
  endedTime = -1

  for i, line in enumerate(f):
    if (line.find("startedTime") == 0):
        startedTime = datetime.datetime.strptime(line[line.find(":")+1:].strip('\n'), '%m/%d/%Y %I:%M:%S %p')
    elif (line.find("endedTime") == 0):
        endedTime = datetime.datetime.strptime(line[line.find(":")+1:].strip('\n'), '%m/%d/%Y %I:%M:%S %p')
        break
    elif (line.find("*") == 0):
      if (line.find("*Lens:") == 0):
        # print(line)
        if line.split(": ")[1].strip('\n') == '90':
          fov_index = 0
        elif line.split(": ")[1].strip('\n') == '110':
          fov_index = 1
        elif line.split(": ")[1].strip('\n') == '130':
          fov_index = 2
        elif line.split(": ")[1].strip('\n') == '150':
          fov_index = 3
        elif line.split(": ")[1].strip('\n') == '170':
          fov_index = 4
    elif (line.find("===") == 0):
      continue
    elif i > 2:
      count += 1
      headAngle_qu[fov_index] = np.append(headAngle_qu[fov_index], [[ float(line.split("%")[0][1:].strip(')').split(",")[0]), 
                                                                      float(line.split("%")[0][1:].strip(')').split(",")[1]), 
                                                                      float(line.split("%")[0][1:].strip(')').split(",")[2]),
                                                                      float(line.split("%")[0][1:].strip(')').split(",")[3]) ]], 
                                                                      axis=0)
      
      headAngle_eu[fov_index] = np.append(headAngle_eu[fov_index], [[ float(line.split("%")[1][1:].strip(')').split(",")[0]), 
                                                                      float(line.split("%")[1][1:].strip(')').split(",")[1]), 
                                                                      float(line.split("%")[1][1:].strip(')').split(",")[2]) ]], 
                                                                      axis=0)

      rightHandPosition[fov_index] = np.append(rightHandPosition[fov_index], [[ float(line.split("%")[2][1:].strip(')').split(",")[0]), 
                                                                                float(line.split("%")[2][1:].strip(')').split(",")[1]), 
                                                                                float(line.split("%")[2][1:].strip(')').split(",")[2]) ]], 
                                                                                axis=0)
      
      leftHandPosition[fov_index] = np.append(leftHandPosition[fov_index], [[ float(line.split("%")[3][1:].strip(')').split(",")[0]), 
                                                                              float(line.split("%")[3][1:].strip(')').split(",")[1]), 
                                                                              float(line.split("%")[3][1:].strip(')').split(",")[2]) ]], 
                                                                              axis=0)
      
      rightHandAngle_qu[fov_index] = np.append(rightHandAngle_qu[fov_index], [[ float(line.split("%")[4][1:].strip(')').split(",")[0]), 
                                                                                float(line.split("%")[4][1:].strip(')').split(",")[1]), 
                                                                                float(line.split("%")[4][1:].strip(')').split(",")[2]),
                                                                                float(line.split("%")[4][1:].strip(')').split(",")[3]) ]], 
                                                                                axis=0)
            
      leftHandAngle_qu[fov_index] = np.append(leftHandAngle_qu[fov_index], [[ float(line.split("%")[5][1:].strip(')').split(",")[0]), 
                                                                              float(line.split("%")[5][1:].strip(')').split(",")[1]), 
                                                                              float(line.split("%")[5][1:].strip(')').split(",")[2]),
                                                                              float(line.split("%")[5][1:].strip(')').split(",")[3]) ]], 
                                                                              axis=0)
      
      rightHandAngle_eu[fov_index] = np.append(rightHandAngle_eu[fov_index], [[ float(line.split("%")[6][1:].strip(')').split(",")[0]), 
                                                                                float(line.split("%")[6][1:].strip(')').split(",")[1]), 
                                                                                float(line.split("%")[6][1:].strip(')').split(",")[2]) ]], 
                                                                                axis=0)
            
      leftHandAngle_eu[fov_index] = np.append(leftHandAngle_eu[fov_index], [[ float(line.split("%")[7][1:].strip(')').split(",")[0]), 
                                                                              float(line.split("%")[7][1:].strip(')').split(",")[1]), 
                                                                              float(line.split("%")[7][1:].strip(')').split(",")[2]) ]], 
                                                                              axis=0)
      
      position[fov_index] = np.append(position[fov_index], [[ float(line.split("%")[8][1:].strip(')\n').split(",")[0]), 
                                                              float(line.split("%")[8][1:].strip(')\n').split(",")[1]), 
                                                              float(line.split("%")[8][1:].strip(')\n').split(",")[2]) ]], 
                                                              axis=0)

  for i in range(5):
    if len(headAngle_qu) > 1:
      headAngle_qu[i] = headAngle_qu[i][1:]
      headAngle_eu[i] = headAngle_eu[i][1:]
      rightHandPosition[i] = rightHandPosition[i][1:]
      leftHandPosition[i] = leftHandPosition[i][1:]
      rightHandAngle_qu[i] = rightHandAngle_qu[i][1:]
      leftHandAngle_qu[i] = leftHandAngle_qu[i][1:]
      rightHandAngle_eu[i] = rightHandAngle_eu[i][1:]
      leftHandAngle_eu[i] = leftHandAngle_eu[i][1:]
      position[i] = position[i][1:]

  for i in range(3):
    if i == 0:
      rightHandAngle_eu_set_90.append(euler_circle_neg2pos(euler_diff_cir_array_1d(rightHandAngle_eu[0][:,1],headAngle_eu[0][:,1])))
      leftHandAngle_eu_set_90.append(euler_circle_neg2pos(euler_diff_cir_array_1d(leftHandAngle_eu[0][:,1], headAngle_eu[0][:,1])))
    elif i == 1:
      rightHandAngle_eu_set_110.append(euler_circle_neg2pos(euler_diff_cir_array_1d(rightHandAngle_eu[1][:,1], headAngle_eu[1][:,1])))
      leftHandAngle_eu_set_110.append(euler_circle_neg2pos(euler_diff_cir_array_1d(leftHandAngle_eu[1][:,1], headAngle_eu[1][:,1])))
    elif i == 2:
      rightHandAngle_eu_set_130.append(euler_circle_neg2pos(euler_diff_cir_array_1d(rightHandAngle_eu[2][:,1], headAngle_eu[2][:,1])))
      leftHandAngle_eu_set_130.append(euler_circle_neg2pos(euler_diff_cir_array_1d(leftHandAngle_eu[2][:,1], headAngle_eu[2][:,1])))
  
  # flatten the list
    flat90R = flatten_list(rightHandAngle_eu_set_90)
    flat90L = flatten_list(leftHandAngle_eu_set_90)
    flat110R = flatten_list(rightHandAngle_eu_set_110)
    flat110L = flatten_list(leftHandAngle_eu_set_110)
    flat130R = flatten_list(rightHandAngle_eu_set_130)
    flat130L = flatten_list(leftHandAngle_eu_set_130)
  

In [ ]:
N = 180

conds = ['90', '110', '130']
RhandsList = [flat90R, flat110R, flat130R]

for i in range(len(conds)):
  theta = np.linspace(0.0, 2 * np.pi, N, endpoint=False)
  radii = np.histogram(RhandsList[i], bins=N, range=[0, 360])[0]
  width = (2*np.pi) / N

  ax = plt.subplot(polar=True)
  bars = ax.bar(theta, radii, width=width, bottom=max(np.histogram(RhandsList[i], bins=N, range=[0, 360])[0])/3, color='slateblue')

  ax.set_title('Euler Right Hand Yaw Direction Distribution in ' + conds[i] + ' FOV')
  ax.set_xlabel('degree [every ' + str(360/N) + r'$\degree$' + ']')
  ax.set_ylabel('frequency', rotation=0, labelpad=-50)

  plt.show()

In [ ]:
N = 180

conds = ['90', '110', '130']
LhandsList = [flat90L, flat110L, flat130L]

for i in range(len(conds)):
  theta = np.linspace(0.0, 2 * np.pi, N, endpoint=False)
  radii = np.histogram(LhandsList[i], bins=N, range=[0, 360])[0]
  width = (2*np.pi) / N

  ax = plt.subplot(polar=True)
  bars = ax.bar(theta, radii, width=width, bottom=max(np.histogram(LhandsList[i], bins=N, range=[0, 360])[0])/3, color='slateblue')

  ax.set_title('Euler Left Hand Yaw Direction Distribution in ' + conds[i] + ' FOV')
  ax.set_xlabel('degree [every ' + str(360/N) + r'$\degree$' + ']')
  ax.set_ylabel('frequency', rotation=0, labelpad=-50)

  plt.show()

In [ ]:
# turn into circular heatmap
# right hands

N = 360  # Number of bins
conds = ['90', '110', '130']
RhandsList = [flat90R, flat110R, flat130R]

for i in range(len(RhandsList)):
    theta = np.linspace(0.0, 2 * np.pi, N, endpoint=False)
    radii, _ = np.histogram(RhandsList[i], bins=N, range=[0, 360])
    theta = np.append(theta, theta[0])  # To make it a closed loop
    radii = np.append(radii, radii[0])  # To make it a closed loop

    ax = plt.subplot(polar=True)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)

    # Create a 2D array of the radii to use pcolormesh
    radii_grid = np.array([radii, radii])
    theta_grid = np.array([theta, theta])

    # Plot the heatmap
    c = ax.pcolormesh(theta_grid, np.array([0, 1]), np.array(radii_grid), shading='auto', cmap='inferno')

    # ax.set_title('Euler Right Hand Yaw Direction Distribution in ' + conds[i] + ' FOV')
    ax.set_xlabel('')
    ax.set_ylabel('', rotation=0, labelpad=-50)

    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.tick_params(axis='both', which='both', length=0)
    ax.grid(False)

    # plt.colorbar(c, ax=ax, pad=0.1, label='Frequency')

    plt.show()

In [ ]:
# turn into circular heatmap
# right hands

N = 360  # Number of bins
conds = ['90', '110', '130']
LhandsList = [flat90L, flat110L, flat130L]

for i in range(len(LhandsList)):
    theta = np.linspace(0.0, 2 * np.pi, N, endpoint=False)
    radii, _ = np.histogram(LhandsList[i], bins=N, range=[0, 360])
    theta = np.append(theta, theta[0])  # To make it a closed loop
    radii = np.append(radii, radii[0])  # To make it a closed loop

    ax = plt.subplot(polar=True)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)

    # Create a 2D array of the radii to use pcolormesh
    radii_grid = np.array([radii, radii])
    theta_grid = np.array([theta, theta])

    # Plot the heatmap
    c = ax.pcolormesh(theta_grid, np.array([0, 1]), np.array(radii_grid), shading='auto', cmap='inferno')

    # ax.set_title('Euler Left Hand Yaw Direction Distribution in ' + conds[i] + ' FOV')
    ax.set_xlabel('')
    ax.set_ylabel('', rotation=0, labelpad=-50)

    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.tick_params(axis='both', which='both', length=0)
    ax.grid(False)

    # plt.colorbar(c, ax=ax, pad=0.1, label='Frequency')

    plt.show()